### Transfers Data Cleaning
Since the 2026 transfer data came from my own scraper, it needs to be cleaned before merging with all other transfer data which was downloaded from a pre-existing GitHub repository.

In [1]:
import pandas as pd
from pathlib import Path

# Define the project folder
project_folder = Path.cwd().parent

# Import the 2026 transfers
df_26 = pd.read_csv(project_folder / "data/raw/transfers/2026.csv")

In [2]:
# 1. Cleaning 2026 data

# Convert the is_loan column to numeric values
df_26["is_loan"] = df_26["is_loan"].astype(int)

# Change season
df_26["season"] = 2026

# Remove entries that are not actual transfers
valid_transfers = (df_26["fee"].fillna("").str.strip().str.lower().str.startswith(("€", "free transfer", "loan fee:", "loan transfer")))

df_26 = df_26[valid_transfers].copy()

# Convert transfer fees and market values
import pandas as pd
import numpy as np
import re

def convert_transfer_value(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    # Transfers with no fee
    if value in ["", "-", "n/a"]:
        return np.nan

    if value in ["free transfer", "loan transfer"]:
        return 0.0

    # Remove descriptive text, leaving the monetary value
    value = value.replace("loan fee:", "").strip()
    value = value.replace("€", "").replace(",", "").strip()

    # Convert millions and thousands
    if value.endswith("m"):
        return float(value[:-1]) * 1_000_000
    elif value.endswith("k"):
        return float(value[:-1]) * 1_000
    else:
        return float(value)

# Convert both columns
for col in ["market_value", "fee"]:
    df_26[col] = df_26[col].apply(convert_transfer_value)

In [5]:
# 2. Cleaning all other data

# Define a function to clean the transfer data
def clean_transfers(year):

    df = pd.read_csv(project_folder/ f"data/raw/transfers/{year}.csv")
    
    transfers_in = df[df["movement"] == "in"].copy()

    transfers_in["fee"] = transfers_in["fee"].fillna(0.0)

    return transfers_in

# Clean all files from 2016 to 2025 and combine into one
all_transfers = pd.concat([clean_transfers(year) for year in range(2016, 2026)],
                          ignore_index=True)

# Add 2026 transfers
all_transfers = pd.concat([all_transfers, df_26], 
                          ignore_index = True)

# Export file
all_transfers.to_csv(project_folder / "data/cleaned/transfers.csv", index = False)

### Player Data Cleaning
Now using data from 2025-present for both the J1 league and the K League 1, we will consolidate all players and seasons into one master data sheet.

In [2]:
# 1. Import the data
import pandas as pd
from pathlib import Path

# Define the project folder
project_folder = Path.cwd().parent

league_year_list = ["j-league-25", "j-league-26", "j-league-26-27",
                     "k-league-25", "k-league-26"]

stats_data = {}
details_data = {}
combined_data = {}

season_dict = {
    "j-league-25": 2025,
    "j-league-26": 2026,
    "j-league-26-27": "2026/27",
    "k-league-25": 2025,
    "k-league-26": 2026
}
league_dict = {
    "j-league-25": "J1 League",
    "j-league-26": "J1 League",
    "j-league-26-27": "J1 League",
    "k-league-25": "K League 1",
    "k-league-26": "K League 1"
}

for league_year in league_year_list:

    stats_path = (project_folder / f"data/raw/players/stats/{league_year}.csv")
    details_path = (project_folder / f"data/raw/players/details/{league_year}.csv")

    stats_data[league_year] = pd.read_csv(stats_path)
    details_data[league_year] = pd.read_csv(details_path)

    print(
        f"{league_year}: "
        f"{len(stats_data[league_year])} stats rows, "
        f"{len(details_data[league_year])} details rows")

    # Merge the season stats
    combined_data[league_year] = pd.merge(stats_data[league_year],
                                          details_data[league_year],
                                          how = "left",
                                          left_on = "player id",
                                          right_on = "id")

    # Create season and league cols
    combined_data[league_year]["season"] = season_dict[league_year]
    combined_data[league_year]["league"] = league_dict[league_year]



j-league-25: 607 stats rows, 607 details rows
j-league-26: 562 stats rows, 562 details rows
j-league-26-27: 510 stats rows, 510 details rows
k-league-25: 400 stats rows, 400 details rows
k-league-26: 364 stats rows, 364 details rows


In [4]:
# 2. Create one master df
master_df = pd.concat(combined_data.values(),
                      ignore_index = True)

In [6]:
# 3. Handle missing positions and define primary position
import ast

positions_map = {"D" : "DC",
                 "M" : "MC",
                 "F" : "ST",
                 "G" : "GK"}

# Apply position mapping for missing values
master_df["positions_detailed"] = (master_df["positions_detailed"].fillna(master_df["position"].map(positions_map)))

# Extract Primary positions from the detailed positions
def safe_literal_eval(x):
    if not isinstance(x, str):
        return x

    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

master_df["positions_detailed"] = (
    master_df["positions_detailed"].apply(safe_literal_eval)
)

invalid_positions = master_df[
    master_df["positions_detailed"].apply(
        lambda x: isinstance(x, str)
    )
]

print(invalid_positions[["player", "positions_detailed"]].head(20))

def get_primary_position(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    elif isinstance(x, str) and x.strip():
        return x
    return None

master_df["primaryPosition"] = (
    master_df["positions_detailed"].apply(get_primary_position)
)

                     player positions_detailed
5                Riku Gunji                 ST
169          Kento Hamasaki                 MC
201            Ichika Maeda                 ST
295           Homare Tokuda                 ST
420  Abdel Hanan Sani Brown                 ST
495           Takanari Endo                 MC
511          Tsuna Kominato                 ST
514            Rinsei Ohata                 MC
543          Shungo Sugiura                 MC
548        Motoki Nishihara                 MC
563        Shimon Kobayashi                 MC
565           Haruto Suzuki                 MC
566           Kantaro Maeda                 ST
568                Yuji Doi                 MC
569         Taiyo Yamaguchi                 ST
571             Hiroto Noda                 DC
583             Sean Kotake                 MC
595           Haruto Hidaka                 DC
596            Hikaru Maeda                 ST
696        Shimon Kobayashi                 MC


In [8]:
# 4. Export the file
master_df.to_csv(project_folder / "data/cleaned/player_data_combined.csv", index = False)